# PDF OCR Extractor v4 - Document Intelligence Edition

**Target: 99% Field-Level Accuracy**

5 Intelligence Layers:
1. Form Schema (UAD 1004)
2. Checkbox Intelligence
3. Field Validation
4. Semantic Normalization
5. Accuracy Metrics

## Step 1: Install Libraries

In [ ]:
print("📦 Installing packages...")
!pip install -q pdf2image pillow opencv-python pytesseract python-docx
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-eng libtesseract-dev
import os
os.environ['TESSDATA_PREFIX'] = '/usr/share/tesseract-ocr/4.00/tessdata/'
print("✅ All packages installed!")
!tesseract --version | head -1

: 

## Step 2: Import Libraries

In [ ]:
import cv2
import numpy as np
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Pt
from datetime import datetime
from google.colab import files
import os
import re
from typing import Dict, List, Tuple, Any, Optional

pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'
print("✅ Libraries imported!")

## Step 3: Load Form Schema & Intelligence Engine

In [ ]:
# Form Schema for UAD 1004
FORM_SCHEMA = {
    "subject": ["property_address", "city", "state", "zip_code", "borrower", "owner_of_record", 
                "county", "legal_description", "assessor_parcel", "tax_year", "re_taxes"],
    "contract": ["contract_price", "date_of_contract", "seller_is_owner"],
    "improvements": ["year_built", "effective_age", "rooms", "bedrooms", "bathrooms", "gla_sqft", "condition"]
}

CHECKBOX_GROUPS = {
    "property_rights": {"options": ["Fee Simple", "Leasehold", "Other"], "exclusive": True},
    "assignment_type": {"options": ["Purchase Transaction", "Refinance Transaction", "Other"], "exclusive": True},
    "occupant": {"options": ["Owner", "Tenant", "Vacant"], "exclusive": True},
    "location": {"options": ["Urban", "Suburban", "Rural"], "exclusive": True},
    "property_values": {"options": ["Increasing", "Stable", "Declining"], "exclusive": True}
}

NORMALIZATION_MAP = {
    "CBS/AVERAGE": "CBS / Average", "CBS/AVG": "CBS / Average",
    "TILE/AVE": "Tile / Average", "TILE/AVG": "Tile / Average",
    "DRYWALL/AVG": "Drywall / Average", "WOOD/AVG": "Wood / Average",
    "C1": "C1 - New", "C2": "C2 - Excellent", "C3": "C3 - Good",
    "C4": "C4 - Average", "C5": "C5 - Fair", "C6": "C6 - Poor"
}

VALIDATION_RULES = {
    "contract_price": {"type": "currency", "min": 10000, "max": 100000000},
    "gla_sqft": {"type": "number", "min": 100, "max": 50000},
    "year_built": {"type": "year", "min": 1800, "max": 2025},
    "bedrooms": {"type": "number", "min": 0, "max": 20},
    "bathrooms": {"type": "number", "min": 0.5, "max": 15}
}

print("✅ Form schema loaded!")
print(f"   Sections: {len(FORM_SCHEMA)}")
print(f"   Checkbox groups: {len(CHECKBOX_GROUPS)}")
print(f"   Validation rules: {len(VALIDATION_RULES)}")

## Step 4: Initialize Document Intelligence Engine

In [ ]:
class FieldValidator:
    def __init__(self, rules):
        self.rules = rules
    
    def validate(self, field, value):
        if field not in self.rules:
            return True, "", value
        rule = self.rules[field]
        if rule["type"] == "currency":
            clean = re.sub(r"[,$\s]", "", str(value))
            try:
                val = float(clean)
                if val < rule["min"] or val > rule["max"]:
                    return False, f"Out of range", val
                return True, "", val
            except:
                return False, "Invalid currency", None
        elif rule["type"] == "number":
            try:
                val = float(str(value).replace(",", ""))
                if val < rule["min"] or val > rule["max"]:
                    return False, f"Out of range", val
                return True, "", val
            except:
                return False, "Invalid number", None
        elif rule["type"] == "year":
            try:
                val = int(str(value).strip())
                if val < rule["min"] or val > rule["max"]:
                    return False, f"Invalid year", val
                return True, "", val
            except:
                return False, "Invalid year", None
        return True, "", value

class SemanticNormalizer:
    def __init__(self, norm_map):
        self.map = {k.upper(): v for k, v in norm_map.items()}
    
    def normalize(self, value):
        if not value:
            return value
        upper = str(value).upper().strip()
        return self.map.get(upper, value)

class AccuracyTracker:
    def __init__(self):
        self.total = 0
        self.valid = 0
        self.corrected = 0
        self.details = {}
        self.critical = ["contract_price", "gla_sqft", "property_address", "borrower"]
    
    def record(self, field, is_valid, was_corrected=False):
        self.total += 1
        if is_valid:
            self.valid += 1
            if was_corrected:
                self.corrected += 1
        self.details[field] = {"valid": is_valid, "corrected": was_corrected, "critical": field in self.critical}
    
    def get_accuracy(self):
        if self.total == 0:
            return 0, 0
        overall = self.valid / self.total * 100
        crit_total = sum(1 for f, d in self.details.items() if d["critical"])
        crit_valid = sum(1 for f, d in self.details.items() if d["critical"] and d["valid"])
        critical = crit_valid / crit_total * 100 if crit_total > 0 else 0
        return overall, critical

validator = FieldValidator(VALIDATION_RULES)
normalizer = SemanticNormalizer(NORMALIZATION_MAP)
accuracy = AccuracyTracker()

print("✅ Intelligence engine initialized!")

## Step 5: Upload PDF File

In [ ]:
print("📤 Please upload your PDF file...")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_filename}")

## Step 6: Convert PDF Page 4 to Image

In [ ]:
print("\n📄 Converting PDF page 4 to image (400 DPI)...")
pages = convert_from_path(pdf_filename, dpi=400, first_page=4, last_page=4)
if len(pages) == 0:
    print("❌ Error: Page 4 not found!")
else:
    page_image = pages[0]
    image_np = np.array(page_image)
    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    print(f"✅ Page 4 extracted: {image_bgr.shape[1]} x {image_bgr.shape[0]} pixels")
    plt.figure(figsize=(12, 16))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title("Page 4 (400 DPI)", fontsize=14)
    plt.axis("off")
    plt.show()

## Step 7: Multi-Variant Preprocessing

In [ ]:
print("\n🔧 Creating preprocessing variants...")
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
p2, p98 = np.percentile(gray, (2, 98))
gray_stretched = cv2.convertScaleAbs(gray, alpha=255.0/max(p98-p2, 1), beta=-p2*255.0/max(p98-p2, 1))
gray_stretched = np.clip(gray_stretched, 0, 255).astype(np.uint8)

variants = {}
_, variants["otsu"] = cv2.threshold(cv2.GaussianBlur(gray_stretched, (3,3), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
variants["adaptive"] = cv2.adaptiveThreshold(gray_stretched, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
variants["sharpened"] = cv2.filter2D(gray_stretched, -1, np.array([[0,-1,0], [-1,5,-1], [0,-1,0]]))
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
variants["clahe"] = clahe.apply(gray)
variants["grayscale"] = gray_stretched

print(f"✅ Created {len(variants)} preprocessing variants")

## Step 8: Intelligent Checkbox Detection

In [ ]:
print("\n☑️ Detecting checkboxes with semantic grouping...")
checkbox_image = image_bgr.copy()
inverted = cv2.bitwise_not(variants["otsu"])
contours, _ = cv2.findContours(inverted, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

checkboxes = []
for contour in contours:
    x, y, w, h = cv2.boundingRect(contour)
    if not (18 <= w <= 55 and 18 <= h <= 55):
        continue
    aspect = float(w) / h
    if not (0.75 <= aspect <= 1.35):
        continue
    area = cv2.contourArea(contour)
    fill = area / (w * h) if w * h > 0 else 0
    if fill < 0.15 or fill > 0.92:
        continue
    eps = 0.05 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, eps, True)
    if 4 <= len(approx) <= 6:
        roi = gray_stretched[y:y+h, x:x+w]
        _, binary = cv2.threshold(roi, 128, 255, cv2.THRESH_BINARY_INV)
        fill_ratio = np.sum(binary > 0) / (w * h) if w * h > 0 else 0
        is_checked = fill_ratio > 0.2
        checkboxes.append({"bbox": (x, y, w, h), "checked": is_checked, "fill": fill_ratio})

# Remove duplicates
unique = []
for cb in checkboxes:
    x1, y1 = cb["bbox"][0], cb["bbox"][1]
    dup = False
    for u in unique:
        x2, y2 = u["bbox"][0], u["bbox"][1]
        if abs(x1-x2) < 12 and abs(y1-y2) < 12:
            dup = True
            break
    if not dup:
        unique.append(cb)

checkboxes = unique
checked_count = sum(1 for cb in checkboxes if cb["checked"])
print(f"✅ Detected {len(checkboxes)} checkboxes ({checked_count} checked, {len(checkboxes)-checked_count} empty)")

for cb in checkboxes:
    x, y, w, h = cb["bbox"]
    color = (0, 255, 0) if cb["checked"] else (255, 0, 0)
    cv2.rectangle(checkbox_image, (x, y), (x+w, y+h), color, 2)

plt.figure(figsize=(14, 18))
plt.imshow(cv2.cvtColor(checkbox_image, cv2.COLOR_BGR2RGB))
plt.title(f"Checkboxes: {len(checkboxes)} (Green=Checked, Red=Empty)", fontsize=14)
plt.axis("off")
plt.show()

## Step 9: Multi-Strategy OCR Extraction

In [ ]:
print("\n📝 Running multi-strategy OCR...")
configs = {
    "best": r"--oem 3 --psm 6 -c preserve_interword_spaces=1",
    "psm3": r"--oem 3 --psm 3",
    "sparse": r"--oem 3 --psm 11"
}

best_score = 0
best_text = ""
best_variant = ""
best_config = ""

for var_name, var_img in variants.items():
    for cfg_name, cfg in configs.items():
        try:
            data = pytesseract.image_to_data(var_img, output_type=pytesseract.Output.DICT, config=cfg)
            confs = [int(c) for i, c in enumerate(data["conf"]) 
                     if str(c) != "-1" and int(c) > 0 and len(str(data["text"][i]).strip()) > 0]
            if len(confs) > 20:
                avg_conf = sum(confs) / len(confs)
                text = pytesseract.image_to_string(var_img, config=cfg)
                score = avg_conf + min(len(text) / 100, 20)
                if score > best_score:
                    best_score = score
                    best_text = text
                    best_variant = var_name
                    best_config = cfg_name
        except:
            pass

data = pytesseract.image_to_data(variants[best_variant], output_type=pytesseract.Output.DICT, config=configs[best_config])
confs = [int(c) for i, c in enumerate(data["conf"]) if str(c) != "-1" and int(c) > 0 and len(str(data["text"][i]).strip()) > 0]
ocr_confidence = sum(confs) / len(confs) if confs else 0

print(f"✅ Best: {best_variant} + {best_config}")
print(f"   OCR Confidence: {ocr_confidence:.1f}%")

## Step 10: Structured Field Extraction

In [ ]:
print("\n🔍 Extracting structured fields...")
extracted = {}

# Regex patterns for key fields
patterns = {
    "contract_price": r"Contract\s*Price[:\s]*\$?([\d,]+)",
    "date_of_contract": r"Date\s*of\s*Contract[:\s]*(\d{1,2}/\d{1,2}/\d{4})",
    "year_built": r"Year\s*Built[:\s]*(\d{4})",
    "effective_age": r"Effective\s*Age[:\s]*(\d+)",
    "gla_sqft": r"(\d{1,2}[,.]?\d{3})\s*Square\s*Feet",
    "rooms": r"(\d+)\s*Rooms",
    "bedrooms": r"(\d+)\s*Bedrooms?",
    "bathrooms": r"([\d.]+)\s*Bath",
    "re_taxes": r"R\.?E\.?\s*Taxes[:\s]*\$?([\d,]+)",
    "zip_code": r"Zip\s*Code[:\s)]*(\d{5})",
    "state": r"State[:\s)]*([A-Z]{2})",
    "assessor_parcel": r"Parcel\s*#?[:\s)]*(\d{2}-\d{4}-\d{3}-\d{4})",
    "census_tract": r"Census\s*Tract[:\s)]*(\d+\.\d+)"
}

for field, pattern in patterns.items():
    match = re.search(pattern, best_text, re.IGNORECASE)
    if match:
        raw_value = match.group(1)
        normalized = normalizer.normalize(raw_value)
        is_valid, error, corrected = validator.validate(field, normalized)
        extracted[field] = {"raw": raw_value, "normalized": normalized, "valid": is_valid, "value": corrected if corrected else normalized}
        accuracy.record(field, is_valid, corrected != raw_value if corrected else False)
        status = "✓" if is_valid else "✗"
        print(f"  {status} {field}: {extracted[field]['value']}")
    else:
        accuracy.record(field, False)
        print(f"  ✗ {field}: NOT FOUND")

print(f"\n✅ Extracted {len(extracted)} / {len(patterns)} fields")

## Step 11: Accuracy Metrics Report

In [ ]:
print("\n📊 Generating accuracy report...")
overall, critical = accuracy.get_accuracy()

print("=" * 60)
print("ACCURACY METRICS REPORT")
print("=" * 60)
print(f"Total Fields Processed: {accuracy.total}")
print(f"Valid Fields: {accuracy.valid}")
print(f"Auto-Corrected: {accuracy.corrected}")
print(f"Failed: {accuracy.total - accuracy.valid}")
print()
print(f"📈 Overall Accuracy: {overall:.1f}%")
print(f"⭐ Critical Field Accuracy: {critical:.1f}%")
print("=" * 60)

print(f"\n☑️ Checkbox Summary:")
print(f"   Total detected: {len(checkboxes)}")
print(f"   Checked: {checked_count}")
print(f"   Empty: {len(checkboxes) - checked_count}")

## Step 12: Generate Output Files

In [ ]:
print("\n📄 Generating output files...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Text file
txt_name = f"page4_extraction_v4_{timestamp}.txt"
with open(txt_name, "w") as f:
    f.write(f"PDF OCR EXTRACTION REPORT v4 - Document Intelligence\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Source: {pdf_filename}\nPage: 4\nDate: {datetime.now()}\n\n")
    f.write(f"OCR Confidence: {ocr_confidence:.1f}%\n")
    f.write(f"Overall Accuracy: {overall:.1f}%\n")
    f.write(f"Critical Accuracy: {critical:.1f}%\n\n")
    f.write("EXTRACTED FIELDS\n" + "-" * 40 + "\n")
    for field, data in extracted.items():
        f.write(f"{field}: {data['value']}\n")
    f.write("\n" + "=" * 70 + "\n")
    f.write("FULL TEXT\n" + "=" * 70 + "\n")
    f.write(best_text)

# Word file
doc = Document()
doc.add_heading("PDF OCR Extraction Report v4", 0)
doc.add_paragraph(f"Source: {pdf_filename}")
doc.add_paragraph(f"OCR Confidence: {ocr_confidence:.1f}% | Overall Accuracy: {overall:.1f}%")
doc.add_heading("Extracted Fields", 1)
for field, data in extracted.items():
    doc.add_paragraph(f"{field}: {data['value']}")
doc.add_heading("Full Text", 1)
doc.add_paragraph(best_text)
docx_name = f"page4_extraction_v4_{timestamp}.docx"
doc.save(docx_name)

print(f"✅ Files created:")
print(f"   📄 {txt_name}")
print(f"   📘 {docx_name}")

print("\n⬇️ Downloading files...")
files.download(txt_name)
files.download(docx_name)

## Step 13: Final Summary

In [ ]:
print("\n" + "*" * 70)
print(" " * 15 + "✅ V4 EXTRACTION COMPLETE")
print("*" * 70)
print(f"\n📊 Results:")
print(f"   OCR Confidence: {ocr_confidence:.1f}%")
print(f"   Overall Accuracy: {overall:.1f}%")
print(f"   Critical Accuracy: {critical:.1f}%")
print(f"   Fields Extracted: {len(extracted)}")
print(f"   Checkboxes: {len(checkboxes)} ({checked_count} checked)")
print(f"\n🔧 Best Config: {best_variant} + {best_config}")
print("\n" + "*" * 70)